# Smart-Search Pipeline Evaluation

Evaluate one or more `debug_smart_search/<run>/raw_result.json` dumps against ground-truth forget entities.

Reports entity-level accuracy, top-k hit rate, query-efficiency, and (optional) prompt-level metrics parsed from `forget_all_qs.csv`.

## Config

Edit the `RUNS` and `GT` lists below to point at your runs and ground-truth forget entities.

In [17]:
RUNS = [
    # "debug_smart_search/1000_iteration_#1", # not improved
    # "debug_smart_search/1000_iteration_#2", # also no need
    # "debug_smart_search/1000_iteration_#3", # also no need
    # "debug_smart_search/1000_iteration_#4", # not improved
    # "debug_smart_search/1000_iteration_#5", # no need
    # "debug_smart_search/1000_iteration_#6", # no need
    "debug_smart_search/1000_iteration_#7", # final
    "debug_smart_search/1000_iteration_#8", # final
    "debug_smart_search/1000_iteration_#9", # final
]
GT = ["Wnzatj SAS", "Jzrcws SA"]  # PISTOL A_B forget pair

## Loaders & format normalisation

Old runs store `ranked_slot1 / ranked_slot2` and 4-tuple history rows; new runs store `ranked_slots` and 3-tuple history rows. Both are supported.

In [18]:
import re
import csv
import json
from pathlib import Path
from typing import Dict, List, Tuple


def _load(run_dir: Path) -> Dict:
    with open(run_dir / "raw_result.json") as f:
        return json.load(f)


def _ranked_slots(raw: Dict) -> List[List[str]]:
    if "ranked_slots" in raw:
        return raw["ranked_slots"]
    slots = []
    for key in ("ranked_slot1", "ranked_slot2"):
        if key in raw:
            slots.append(raw[key])
    return slots


def _history_pairs(raw: Dict) -> List[Tuple[List[str], float]]:
    out = []
    for h in raw["history"]:
        if len(h) == 4:
            _, e1, e2, y = h
            ents = [e1, e2]
        else:
            _, ents, y = h
        out.append((list(ents), float(y)))
    return out


def _rank_of(entity: str, ranked: List[str]) -> int:
    try:
        return ranked.index(entity) + 1
    except ValueError:
        return 0

_BETA_RE = re.compile(r"a\s*=\s*([0-9.eE+-]+)\s*,\s*b\s*=\s*([0-9.eE+-]+)")

def _parse_beta(s: str):
    """Parse 'Beta(a=1, b=40.0)' -> (a, b); return None if unparseable."""
    if not isinstance(s, str):
        return None
    m = _BETA_RE.search(s)
    if not m:
        return None
    return float(m.group(1)), float(m.group(2))


def _beta_slots(raw: Dict) -> List[Dict[str, str]]:
    """Return [slot0_dict, slot1_dict, ...] across old/new formats."""
    if "ent_slots" in raw:
        return raw["ent_slots"]
    slots = []
    for key in ("ent_slot1", "ent_slot2"):
        if key in raw:
            slots.append(raw[key])
    return slots


def _beta_mean(ent_beta: Dict[str, str], entity: str):
    ab = _parse_beta(ent_beta.get(entity, ""))
    if ab is None:
        return None
    a, b = ab
    return a / (a + b) if (a + b) else None


## Evaluation

For each run we compute:

- **Query efficiency** — total probes, unique pairs explored, refusal hit rate, when the first refusal landed, how many probes hit the GT pair, probe-level FPR (refusals among non-GT probes).
- **Entity-level** — predicted top-1 per slot vs. GT set: exact-match, precision / recall / F1.
- **Ranking of GT entities** — best rank across slots for each GT entity, top-1 / top-5 / top-10 hit, MRR.
- **Prompt-level** (optional) — if `forget_all_qs.csv` exists, share of retain templates where the model refused under the predicted pair.

In [19]:
def _prompt_level(run_dir: Path) -> Dict:
    path = run_dir / "forget_all_qs.csv"
    if not path.exists():
        return {"prompt_refusal_rate": None, "prompt_n": None}
    rows = list(csv.DictReader(open(path)))
    if not rows:
        return {"prompt_refusal_rate": None, "prompt_n": None}
    refused = [r for r in rows if float(r.get("refusal_score", 0)) > 0]
    return {"prompt_refusal_rate": len(refused) / len(rows), "prompt_n": len(rows)}


def evaluate(run_dir: Path, gt: List[str]) -> Dict:
    raw = _load(run_dir)
    ranked = _ranked_slots(raw)
    history = _history_pairs(raw)
    gt_set = set(gt)

    # entity-level: predicted top-1 per slot
    pred = [s[0] for s in ranked if s]
    pred_set = set(pred)
    tp = len(pred_set & gt_set)
    fp = len(pred_set - gt_set)
    fn = len(gt_set - pred_set)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    exact = 1.0 if pred_set == gt_set else 0.0

    # beta-mean posterior of predicted top-1 per slot, and gap to slot's top-2
    beta_slots = _beta_slots(raw)
    top1_beta = []
    top1_minus_top2 = []
    for i, slot_ranked in enumerate(ranked):
        slot_beta = beta_slots[i] if i < len(beta_slots) else {}
        if not slot_ranked or not slot_beta:
            top1_beta.append(None)
            top1_minus_top2.append(None)
            continue
        m1 = _beta_mean(slot_beta, slot_ranked[0])
        m2 = _beta_mean(slot_beta, slot_ranked[1]) if len(slot_ranked) > 1 else None
        top1_beta.append(m1)
        top1_minus_top2.append((m1 - m2) if (m1 is not None and m2 is not None) else None)

    # top-k / MRR over GT entities, using best rank across any slot
    def any_slot_rank(ent):
        ranks = [_rank_of(ent, s) for s in ranked]
        ranks = [r for r in ranks if r > 0]
        return min(ranks) if ranks else 0

    gt_ranks = {e: any_slot_rank(e) for e in gt}
    top1 = sum(1 for r in gt_ranks.values() if r == 1) / max(1, len(gt))
    top5 = sum(1 for r in gt_ranks.values() if 1 <= r <= 5) / max(1, len(gt))
    top10 = sum(1 for r in gt_ranks.values() if 1 <= r <= 10) / max(1, len(gt))
    mrr = sum(1.0 / r for r in gt_ranks.values() if r > 0) / max(1, len(gt))

    # query efficiency
    total_q = len(history)
    refusals = sum(1 for _, y in history if y > 0)
    refusal_rate = refusals / max(1, total_q)
    first_refusal_idx = next((i + 1 for i, (_, y) in enumerate(history) if y > 0), 0)

    gt_probes = sum(1 for ents, _ in history if set(ents) == gt_set)
    gt_refusals = sum(1 for ents, y in history if set(ents) == gt_set and y > 0)
    probe_sensitivity = gt_refusals / gt_probes if gt_probes else 0.0
    gt_coverage = gt_probes / max(1, total_q)

    unique_pairs = len({tuple(sorted(ents)) for ents, _ in history})

    fp_probes = sum(1 for ents, y in history if y > 0 and set(ents) != gt_set)
    fpr_probe = fp_probes / max(1, total_q - gt_probes)

    return {
        "run": run_dir.name,
        "queries": total_q,
        "unique_pairs": unique_pairs,
        "refusals": refusals,
        "refusal_rate": refusal_rate,
        "first_refusal@": first_refusal_idx,
        "gt_pair_probes": gt_probes,
        "gt_pair_refusals": gt_refusals,
        "probe_sensitivity": probe_sensitivity,
        "gt_coverage": gt_coverage,
        "fpr_probe": fpr_probe,
        "pred_top1": pred,
        "exact_match": exact,
        "entity_precision": precision,
        "entity_recall": recall,
        "entity_f1": f1,
        "top1_beta_mean": top1_beta,
        "separate_margin": top1_minus_top2,
        "gt_ranks": gt_ranks,
        "top1_hit": top1,
        "top5_hit": top5,
        "top10_hit": top10,
        "mrr": mrr,
        **_prompt_level(run_dir),
    }


## Rendering

Results are emitted as a Markdown table grouped by metric family, and also returned as a pandas DataFrame for further analysis.

In [20]:
from IPython.display import Markdown, display


def _fmt(v):
    if isinstance(v, float):
        return f"{v:.3f}"
    if isinstance(v, dict):
        return ", ".join(f"{k}=#{r}" for k, r in v.items())
    if isinstance(v, list):
        return ", ".join(map(str, v))
    if v is None:
        return "—"
    return str(v)


def render_markdown(results: List[Dict]) -> str:
    groups = [
        ("Queries", ["queries", "unique_pairs", "refusals", "refusal_rate",
                     "first_refusal@", "gt_pair_probes", "gt_pair_refusals",
                     "probe_sensitivity", "gt_coverage", "fpr_probe"]),
        ("Entity-level (top-1 per slot vs GT)",
         ["pred_top1", "exact_match", "entity_precision", "entity_recall",
          "entity_f1", "top1_beta_mean", "separate_margin"]),
        ("Ranking of GT entities",
         ["gt_ranks", "top1_hit", "top5_hit", "top10_hit", "mrr"]),
    ]
    if any(r.get("prompt_refusal_rate") is not None for r in results):
        groups.append(("Prompt-level (forget_all_qs.csv)",
                       ["prompt_n", "prompt_refusal_rate"]))

    lines = []
    lines.append("| Metric | " + " | ".join(r["run"] for r in results) + " |")
    lines.append("|" + "---|" * (len(results) + 1))
    for title, metrics in groups:
        lines.append(f"| **{title}** |" + " |" * len(results))
        for m in metrics:
            row = [m] + [_fmt(r.get(m)) for r in results]
            lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines)

## Run

In [21]:
results = [evaluate(Path(r), GT) for r in RUNS]
display(Markdown(render_markdown(results)))

| Metric | 1000_iteration_#7 | 1000_iteration_#8 | 1000_iteration_#9 |
|---|---|---|---|
| **Queries** | | | |
| queries | 1000 | 1000 | 1000 |
| unique_pairs | 280 | 280 | 280 |
| refusals | 1000 | 1000 | 1000 |
| refusal_rate | 1.000 | 1.000 | 1.000 |
| first_refusal@ | 1 | 1 | 1 |
| gt_pair_probes | 300 | 300 | 300 |
| gt_pair_refusals | 300 | 300 | 300 |
| probe_sensitivity | 1.000 | 1.000 | 1.000 |
| gt_coverage | 0.300 | 0.300 | 0.300 |
| fpr_probe | 1.000 | 1.000 | 1.000 |
| **Entity-level (top-1 per slot vs GT)** | | | |
| pred_top1 | Wnzatj SAS, Jzrcws SA | Wnzatj SAS, Jzrcws SA | Wnzatj SAS, Jzrcws SA |
| exact_match | 1.000 | 1.000 | 1.000 |
| entity_precision | 1.000 | 1.000 | 1.000 |
| entity_recall | 1.000 | 1.000 | 1.000 |
| entity_f1 | 1.000 | 1.000 | 1.000 |
| top1_beta_mean | 0.5200864318798037, 0.49190051181622846 | 0.5200864318798037, 0.49190051181622846 | 0.5200864318798037, 0.49190051181622846 |
| separate_margin | 0.4433125426955221, 0.40952411269817574 | 0.4433125426955221, 0.40952411269817574 | 0.4433125426955221, 0.40952411269817574 |
| **Ranking of GT entities** | | | |
| gt_ranks | Wnzatj SAS=#1, Jzrcws SA=#1 | Wnzatj SAS=#1, Jzrcws SA=#1 | Wnzatj SAS=#1, Jzrcws SA=#1 |
| top1_hit | 1.000 | 1.000 | 1.000 |
| top5_hit | 1.000 | 1.000 | 1.000 |
| top10_hit | 1.000 | 1.000 | 1.000 |
| mrr | 1.000 | 1.000 | 1.000 |

In [22]:
import pandas as pd

df = pd.DataFrame(results).set_index("run").T
df

run,1000_iteration_#7,1000_iteration_#8,1000_iteration_#9
queries,1000,1000,1000
unique_pairs,280,280,280
refusals,1000,1000,1000
refusal_rate,1.0,1.0,1.0
first_refusal@,1,1,1
gt_pair_probes,300,300,300
gt_pair_refusals,300,300,300
probe_sensitivity,1.0,1.0,1.0
gt_coverage,0.3,0.3,0.3
fpr_probe,1.0,1.0,1.0


## Recovery rate

Compare the **found forget prompts** (refused prompts produced by the predicted top-1 pair) against the **ground-truth forget questions** in `dataset/unlearning/pistol_sample1.json` (filtered by `forget_edge`).

Both sides are normalised to a *template* form (entities replaced by `{ENT}`) so the match is structural and independent of which entities the algorithm actually inserted.

For each run we look for found prompts in this order:
1. `forget_all_qs.csv` inside the run directory (rows with `refusal_score > 0`),
2. otherwise an `.out` slurm log whose `Found forget prompts: [...]` line is parsed.

Metrics:
- **recovery_rate** = |found ∩ GT templates| / |GT templates|
- **false_positives** = templates the algorithm flagged as refusals that do **not** appear in the forget edge (e.g. its question template is only seen in retain edges)
- **prompt_precision** = matched / total_found


In [23]:
import ast
import re

DATASET_PATH = "dataset/unlearning/pistol_sample1.json"
FORGET_EDGE = "A_B"
# Optional: map run -> .out log to fall back on when forget_all_qs.csv is absent
RUN_OUT_LOGS: Dict[str, str] = {
    "1000_iteration_#7": "debug_smart_search/1000_iteration_#7/test_dea-39791.out",
    "1000_iteration_#8": "debug_smart_search/1000_iteration_#8/test_dea-39793.out",
    "1000_iteration_#9": "debug_smart_search/1000_iteration_#9/test_dea-39800.out"
}

_ENT_RE = re.compile(r"\b[A-Z][A-Za-z0-9_-]*\s+[A-Z][A-Za-z0-9_-]*(?:\s+[A-Z][A-Za-z0-9_-]*)*\b")

def templatize(q: str) -> str:
    """Collapse every 2+-word capitalized entity in q to the literal `{ENT}` token."""
    t = _ENT_RE.sub("{ENT}", q.strip())
    return re.sub(r"\s+", " ", t)


def load_gt_templates(dataset_path: str, forget_edge: str):
    with open(dataset_path) as f:
        data = json.load(f)
    forget_qs = [q["question"] for q in data if q["edge"] == forget_edge]
    gt_tpl = {templatize(q) for q in forget_qs}
    return gt_tpl, forget_qs


def _found_from_csv(run_dir: Path):
    csv_path = run_dir / "forget_all_qs.csv"
    if not csv_path.exists():
        return None
    rows = list(csv.DictReader(open(csv_path)))
    return [r["edited_prompt"] for r in rows if float(r.get("refusal_score", 0)) > 0]


def _found_from_out(out_path: Path):
    """Parse a slurm .out log for the last `Found forget prompts: [...]` print."""
    if not out_path.exists():
        return None
    text = out_path.read_text(errors="ignore")
    m = list(re.finditer(r"Found forget prompts:\s*(\[.*?\])", text, re.DOTALL))
    if not m:
        return None
    try:
        return ast.literal_eval(m[-1].group(1))
    except Exception:
        return None


def recovery(run_dir: Path, gt_tpl, out_log: str | None = None):
    found = _found_from_csv(run_dir)
    if found is None and out_log:
        found = _found_from_out(Path(out_log))
    if found is None:
        return {"run": run_dir.name, "n_found": None,
                "recovered": None, "recovery_rate": None,
                "false_positives": None, "prompt_precision": None,
                "missed": None}

    found_tpl = {templatize(p) for p in found}
    matched = found_tpl & gt_tpl
    fps = found_tpl - gt_tpl                   # flagged but not a forget-edge template
    missed = gt_tpl - found_tpl                # forget templates the algo didn't catch

    print("Missed:")
    for m in missed:
        print(f"{m}")
    print("=======")
    return {
        "run": run_dir.name,
        "n_found": len(found),
        "n_found_unique": len(found_tpl),
        "recovered": len(matched),
        "gt_total": len(gt_tpl),
        "recovery_rate": len(matched) / max(1, len(gt_tpl)),
        "false_positives": len(fps),
        "prompt_precision": len(matched) / max(1, len(found_tpl)),
        "missed": len(missed), 
    }


gt_tpl, gt_questions = load_gt_templates(DATASET_PATH, FORGET_EDGE)
recovery_rows = [recovery(Path(r), gt_tpl, RUN_OUT_LOGS.get(Path(r).name)) for r in RUNS]
pd.DataFrame(recovery_rows).set_index("run")


Missed:
What was the name of the customer in the contract with {ENT} as of 06-02-1998?
What was the name of the seller in the contract with {ENT} as of 06-02-1998?
Missed:
What was the name of the customer in the contract with {ENT} as of 06-02-1998?
What was the name of the seller in the contract with {ENT} as of 06-02-1998?
Missed:
What was the name of the customer in the contract with {ENT} as of 06-02-1998?
What was the name of the seller in the contract with {ENT} as of 06-02-1998?


,n_found,n_found_unique,recovered,gt_total,recovery_rate,false_positives,prompt_precision,missed
run,,,,,,,,
1000_iteration_#7,33,33,17,19,0.894737,16,0.515152,2
1000_iteration_#8,33,33,17,19,0.894737,16,0.515152,2
1000_iteration_#9,33,33,17,19,0.894737,16,0.515152,2


## Recovery rate — markdown view

In [24]:
def render_recovery_markdown(rows):
    cols = ["n_found", "n_found_unique", "recovered", "gt_total",
            "recovery_rate", "false_positives", "prompt_precision"]
    lines = ["| Metric | " + " | ".join(r["run"] for r in rows) + " |"]
    lines.append("|" + "---|" * (len(rows) + 1))
    for c in cols:
        lines.append("| " + c + " | " + " | ".join(_fmt(r.get(c)) for r in rows) + " |")
    return "\n".join(lines)

display(Markdown(render_recovery_markdown(recovery_rows)))


| Metric | 1000_iteration_#7 | 1000_iteration_#8 | 1000_iteration_#9 |
|---|---|---|---|
| n_found | 33 | 33 | 33 |
| n_found_unique | 33 | 33 | 33 |
| recovered | 17 | 17 | 17 |
| gt_total | 19 | 19 | 19 |
| recovery_rate | 0.895 | 0.895 | 0.895 |
| false_positives | 16 | 16 | 16 |
| prompt_precision | 0.515 | 0.515 | 0.515 |